Merging script

Loading Python libraries

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

Pathing script

In [2]:
# Define the base directory where all your Eurostat files are stored
BASE_DIR = os.path.expanduser(r"~\OneDrive\Desktop\TIL Programming\6020 Group project\Project data_Freight") # change this to your folder path

# Build file paths safely using os.path.join
files = {"rail_go_total": os.path.join(BASE_DIR, "rail_go_total__custom_18309054_linear_2_0.csv"),"rail_if_line_na": os.path.join(BASE_DIR, "rail_if_line_na__custom_18324484_linear_2_0.csv"),"tran_hv_frmod": os.path.join(BASE_DIR, "tran_hv_frmod__custom_18309026_linear_2_0.csv"),"ttr00006": os.path.join(BASE_DIR, "ttr00006__custom_18309048_linear_2_0.csv"),"rail_go_consgmt": os.path.join(BASE_DIR, "rail_go_consgmt__custom_18308929_linear_2_0.csv"),"rail_go_grpgood": os.path.join(BASE_DIR, "rail_go_grpgood__custom_18309135_linear_2_0.csv")}

# EU27 country list (2025 definition)
EU27 = ["BE","BG","CZ","CH","DK","DE","EE","IE","EL","ES","FR","HR","IT","CY","LV","LT","LU","HU","MT","NL","AT","PL","PT","RO","SI","SK","FI","SE"]

Appending rail_go_total

In [3]:
# --- Load the first dataset (rail_go_total) and initialize df_master ---
file_path = files["rail_go_total"]
df_rail_go_total = pd.read_csv(file_path)

# Select relevant columns
cols_to_keep = ["geo", "unit", "OBS_VALUE", "TIME_PERIOD"]
df_total = df_rail_go_total[cols_to_keep]

# ---  Filter to EU27 countries ---
df_total = df_total[df_total["geo"].isin(EU27)]

# --- Filter out rows where unit == 'MIO_TKM' ---
df_total = df_total[df_total["unit"] != "MIO_TKM"]

# --- Rename OBS_VALUE column ---
df_total = df_total.rename(columns={"OBS_VALUE": "Observed_freight_total_THST"})

# --- Initialize df_master ---
df_master = df_total.drop(columns=["unit"], errors="ignore").copy()

print(df_master.head(5))

    geo  Observed_freight_total_THST  TIME_PERIOD
540  AT                     121579.0         2008
541  AT                      98887.0         2009
542  AT                     107670.0         2010
543  AT                     107587.0         2011
544  AT                     100452.0         2012


Appending rail_if_line_na

In [4]:
# --- Step 1: Read rail_if_line_na from your files dictionary ---
file_path = files["rail_if_line_na"]
df_rail_if_line_na = pd.read_csv(file_path)

# --- Step 2: Select only relevant columns ---
cols_to_keep = ["geo", "unit", "OBS_VALUE", "TIME_PERIOD"]
df_if_line_na = df_rail_if_line_na[cols_to_keep]

# --- Step 3: Rename OBS_VALUE to 'Network_length' ---
df_if_line_na = df_if_line_na.rename(columns={"OBS_VALUE": "Network_length_KM"})

# --- Step 4: Filter to EU27 countries ---
df_if_line_na = df_if_line_na[df_if_line_na["geo"].isin(EU27)]

# --- Step 5: Merge on geo + TIME_PERIOD, keep those from master only ---
df_master = pd.merge(df_master, df_if_line_na, on=["geo", "TIME_PERIOD"], how="left", suffixes=("", "_new"))

# Drop any duplicate geo/TIME_PERIOD columns from the new dataset after merge
for col in ["geo_new", "unit", "TIME_PERIOD_new"]:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

print(df_master.head(1))

  geo  Observed_freight_total_THST  TIME_PERIOD  Network_length_KM
0  AT                     121579.0         2008             5693.0


Appending tran_hv_frmod

In [5]:
# --- Step 1: Read tran_hv_frmod from your files dictionary ---
file_path = files["tran_hv_frmod"]
df_tran_hv_frmod = pd.read_csv(file_path)

# --- Step 2: Keep relevant columns ---
cols_to_keep = ["geo", "TIME_PERIOD", "unit", "tra_mode", "OBS_VALUE"]
df_frmod = df_tran_hv_frmod[cols_to_keep]

# --- Step 3: Filter to only railway transport mode ---
df_frmod = df_frmod[df_frmod["tra_mode"].str.lower().str.contains("rail")]

# --- Step 4: Rename OBS_VALUE to Modal_Share_PCT ---
df_frmod = df_frmod.rename(columns={"OBS_VALUE": "Modal_Share_PCT"})

# --- Step 5: Filter to EU27 countries ---
df_frmod = df_frmod[df_frmod["geo"].isin(EU27)]

# --- Step 6: Merge on geo + TIME_PERIOD ---
df_master = pd.merge(df_master, df_frmod[["geo", "TIME_PERIOD", "Modal_Share_PCT"]], on=["geo", "TIME_PERIOD"], how="left",suffixes=("", "_new"))

# --- Step 7: Drop any duplicate geo/TIME_PERIOD columns ---
for col in ["geo_new", "TIME_PERIOD_new"]:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

print(df_master.head(1))

  geo  Observed_freight_total_THST  TIME_PERIOD  Network_length_KM  \
0  AT                     121579.0         2008             5693.0   

   Modal_Share_PCT  
0             33.6  


Appending Rail_go__csgmnt

In [6]:
# --- Step 1: Read rail_go_consgmt from your files dictionary ---
file_path = files["rail_go_consgmt"]
df_rail_go_consgmt = pd.read_csv(file_path)

# --- Step 2: Keep only relevant columns ---
cols_to_keep = ["geo", "TIME_PERIOD", "unit", "Type of consignment", "OBS_VALUE"]
df_consgmt = df_rail_go_consgmt[cols_to_keep]

# --- Step 3: Filter out rows where unit == 'MIO_TKM' ---
df_consgmt = df_consgmt[df_consgmt["unit"] != "MIO_TKM"]

# --- Step 4: Filter to EU27 countries ---
df_consgmt = df_consgmt[df_consgmt["geo"].isin(EU27)]

# --- Step 5: Pivot 'Type of consignment' into separate columns ---
df_consgmt_wide = df_consgmt.pivot_table(index=["geo", "TIME_PERIOD"], columns="Type of consignment", values="OBS_VALUE", aggfunc="first").reset_index()

# --- Step 6: Rename columns for clarity ---
df_consgmt_wide.columns = ["geo" if col == "geo" else "TIME_PERIOD" if col == "TIME_PERIOD" else f"Consignment_{col.strip().replace(' ', '_').lower()}_THS_T"for col in df_consgmt_wide.columns]

# --- Step 7: Merge with master on geo + TIME_PERIOD ---
df_master = pd.merge(df_master, df_consgmt_wide, on=["geo", "TIME_PERIOD"], how="left", suffixes=("", "_new"))

# --- Step 8: Drop any duplicate key columns from the merge ---
for col in ["geo_new", "TIME_PERIOD_new"]:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

print(df_master.head(1))

  geo  Observed_freight_total_THST  TIME_PERIOD  Network_length_KM  \
0  AT                     121579.0         2008             5693.0   

   Modal_Share_PCT  Consignment_full_train_load_THS_T  \
0             33.6                                NaN   

   Consignment_full_wagon_load_THS_T  Consignment_total_THS_T  
0                                NaN                      NaN  


Append Rail_go_grpgood

In [7]:
# --- Step 1: Read rail_go_grpgood from your files dictionary ---
file_path = files["rail_go_grpgood"]
df_rail_go_grpgood = pd.read_csv(file_path)

# --- Step 2: Keep only relevant columns ---
cols_to_keep = ["geo", "TIME_PERIOD", "unit", "nst07", "OBS_VALUE"]
df_grpgood = df_rail_go_grpgood[cols_to_keep]

# --- Step 3: Filter to EU27 countries ---
df_grpgood = df_grpgood[df_grpgood["geo"].isin(EU27)]

# --- Step 4: Pivot NST categories into separate columns ---
df_grpgood_wide = df_grpgood.pivot_table(index=["geo", "TIME_PERIOD"], columns="nst07", values="OBS_VALUE", aggfunc="first").reset_index()

# --- Step 5: Rename NST columns with '_MIO_TKM' suffix ---
df_grpgood_wide.columns = [f"NST_{col}_MIO_TKM" if col not in ["geo", "TIME_PERIOD"] else col
    for col in df_grpgood_wide.columns]

# --- Step 6: Merge with master on geo + TIME_PERIOD ---
df_master = pd.merge(df_master, df_grpgood_wide, on=["geo", "TIME_PERIOD"], how="left", suffixes=("", "_new"))

# --- Step 7: Drop any duplicate geo/TIME_PERIOD columns from new dataset ---
for col in ["geo_new", "TIME_PERIOD_new"]:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

print(df_master.head(1))

  geo  Observed_freight_total_THST  TIME_PERIOD  Network_length_KM  \
0  AT                     121579.0         2008             5693.0   

   Modal_Share_PCT  Consignment_full_train_load_THS_T  \
0             33.6                                NaN   

   Consignment_full_wagon_load_THS_T  Consignment_total_THS_T  \
0                                NaN                      NaN   

   NST_GT01_MIO_TKM  NST_GT02_MIO_TKM  ...  NST_GT12_MIO_TKM  \
0               NaN               NaN  ...               NaN   

   NST_GT13_MIO_TKM  NST_GT14_MIO_TKM  NST_GT15_MIO_TKM  NST_GT16_MIO_TKM  \
0               NaN               NaN               NaN               NaN   

   NST_GT17_MIO_TKM  NST_GT18_MIO_TKM  NST_GT19_MIO_TKM  NST_GT20_MIO_TKM  \
0               NaN               NaN               NaN               NaN   

   NST_TOTAL_MIO_TKM  
0                NaN  

[1 rows x 29 columns]


Removing duplicate rows and Re-index

In [8]:
# --- Step 1: Drop duplicate rows based on all columns ---
df_master = df_master.drop_duplicates()

# --- Step 2: Ensure each geo-year pair is unique ---
df_master = df_master.drop_duplicates(subset=["geo", "TIME_PERIOD"])

# --- Step 3: Reset index for cleanliness ---
df_master = df_master.reset_index(drop=True)

Visual inspection

In [9]:
# Display the entire DataFrame (be careful if it's large)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(df_master)

,geo,Observed_freight_total_THST,TIME_PERIOD,Network_length_KM,Modal_Share_PCT,Consignment_full_train_load_THS_T,Consignment_full_wagon_load_THS_T,Consignment_total_THS_T,NST_GT01_MIO_TKM,NST_GT02_MIO_TKM,NST_GT03_MIO_TKM,NST_GT04_MIO_TKM,NST_GT05_MIO_TKM,NST_GT06_MIO_TKM,NST_GT07_MIO_TKM,NST_GT08_MIO_TKM,NST_GT09_MIO_TKM,NST_GT10_MIO_TKM,NST_GT11_MIO_TKM,NST_GT12_MIO_TKM,NST_GT13_MIO_TKM,NST_GT14_MIO_TKM,NST_GT15_MIO_TKM,NST_GT16_MIO_TKM,NST_GT17_MIO_TKM,NST_GT18_MIO_TKM,NST_GT19_MIO_TKM,NST_GT20_MIO_TKM,NST_TOTAL_MIO_TKM
0,AT,121579.0,2008,5693.000,33.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AT,98887.0,2009,5693.000,32.0,NaN,NaN,NaN,1890.0,740.0,1247.0,263.0,1.0,1555.0,1394.0,773.0,312.0,1081.0,81.0,980.0,4.0,839.0,11.0,239.0,4.0,23.0,3918.0,0.0,15355.0
2,AT,107670.0,2010,5828.000,33.0,NaN,NaN,NaN,2039.0,681.0,1633.0,263.0,1.0,1632.0,1449.0,833.0,325.0,1291.0,68.0,1042.0,5.0,1162.0,13.0,420.0,127.0,255.0,4649.0,0.0,17886.0
3,AT,107587.0,2011,5500.000,33.1,NaN,NaN,NaN,2017.0,733.0,1668.0,288.0,1.0,1520.0,1566.0,870.0,263.0,1424.0,71.0,1080.0,8.0,1242.0,14.0,419.0,167.0,330.0,4607.0,0.0,18288.0
4,AT,100452.0,2012,5566.000,32.7,NaN,NaN,NaN,1786.0,712.0,1599.0,302.0,2.0,1483.0,1638.0,902.0,241.0,1346.0,63.0,1031.0,7.0,1102.0,11.0,408.0,188.0,370.0,4079.0,0.0,17269.0
5,AT,96449.0,2013,5531.000,32.1,NaN,NaN,NaN,1649.0,868.0,1789.0,302.0,2.0,1418.0,1596.0,837.0,263.0,1536.0,66.0,1041.0,8.0,1242.0,7.0,204.0,0.0,38.0,5147.0,0.0,18012.0
6,AT,100267.0,2014,5531.000,33.1,NaN,NaN,NaN,1664.0,915.0,1900.0,230.0,2.0,1339.0,1741.0,951.0,302.0,1629.0,60.0,1298.0,7.0,1283.0,9.0,187.0,0.0,3.0,6004.0,0.0,19522.0
7,AT,100163.0,2015,5522.000,32.5,NaN,NaN,NaN,1364.0,938.0,1777.0,223.0,1.0,1366.0,1673.0,1027.0,301.0,1648.0,56.0,1468.0,8.0,1191.0,10.0,173.0,0.0,4.0,6506.0,0.0,19736.0
8,AT,102835.0,2016,5491.000,32.1,NaN,NaN,NaN,1671.0,908.0,1877.0,226.0,1.0,1399.0,1448.0,1043.0,270.0,1776.0,136.0,1439.0,8.0,1243.0,11.0,217.0,0.0,41.0,7646.0,0.0,21361.0
9,AT,107579.0,2017,5527.000,31.9,NaN,NaN,NaN,1755.0,932.0,1920.0,251.0,1.0,1392.0,1521.0,1051.0,247.0,1959.0,190.0,1209.0,8.0,1334.0,11.0,228.0,0.0,38.0,8208.0,0.0,22256.0


DF -> .CSV

In [10]:
# Define the export path to your project folder
output_path = os.path.expanduser(r"~\OneDrive\Desktop\TIL Programming\6020 Group project\Project data_Freight\merged_eurostat_clean_V2.csv")

# Save as CSV
df_master.to_csv(output_path, index=False, encoding="utf-8")